<a href="https://colab.research.google.com/github/QUANHONGLE/CS421-emotion-prediction/blob/main/Q2_InContext_Learning_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade transformers accelerate

In [2]:
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft

In [3]:
!pip install evaluate

In [4]:
!pip install rouge_score
!pip install bert-score
!pip install sacrebleu

In [5]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, pipeline
import evaluate
import re
import random

In [6]:
import gc
torch.cuda.empty_cache()
gc.collect()

289

In [7]:
# model set up
torch.random.manual_seed(0)

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")
tokenizer.pad_token = tokenizer.eos_token

config = AutoConfig.from_pretrained("microsoft/phi-2")
config.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2", # PHI 3 mini kept giving errors so switch to phi-2
    config=config,
    torch_dtype=torch.float16,
)

model = model.to("cuda")
print(next(model.parameters()).device)  # should print cuda:0

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0,  # force to use gpu
)

generation_args = {
    "max_new_tokens": 500,  #changed from 150 to 180
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

cuda:0


In [8]:
def load_convo(df):
  convo = []
  # dev data set has speaker id instead of speaker
  speaker_col = "speaker" if "speaker" in df.columns else "speaker_id"

  for conv_id, group in df.groupby("conversation_id"):
    group = group.sort_values("turn_id")
    turns = []
    for col, row in group.iterrows():
      turns.append({"speaker":row[speaker_col], "utterance": row["text"]})

    convo.append({"id":conv_id, "turns": turns})
  return convo

In [9]:
def get_turn(turns, start_num=1):
    turn_lines = []
    # getting turn count and utterance
    for i, turn in enumerate(turns, start=start_num):
        turn_lines.append(f"Turn {i}: {turn['utterance']}")
    return "\n".join(turn_lines)

def build_prompts(train_data, num_shots):
    prompts = []
    sampled = random.sample(train_data, num_shots)

    for convo in sampled:
        context = convo["turns"][:5]
        target = convo["turns"][5:15]
        prompts.append({
            "context": get_turn(context, start_num=1),
            "target": get_turn(target, start_num=6)
        })

    return prompts

def build_ICL_prompts(few_shot_examples, query):
    prompt = "Given a conversation history of 5 turns, please generate the next 10 turns.\n"

    for example in few_shot_examples:
        prompt += f"Conversation history:\n{example['context']}\n\nNext 10 turns:\n{example['target']}\n\n"
        prompt += "---\n\n"

    prompt += f"Conversation history:\n{query}\n\nNext 10 turns:\n"
    return prompt
# trying to get formatting for debugging
def get_turns(text):
    matches = re.findall(r'(?:Turn|turn|Talk|talk)\s*(\d+)\s*:\s*(.+)', text)

    turns = []
    for num, content in matches:
        num = int(num)
        if 6 <= num <= 15:
            turns.append(content.strip())
        if len(turns) == 10:
            break

    return turns

def safe_parse(text):
    turns = get_turns(text)

    # in case format didn't work this will default
    if len(turns) == 0:
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        return lines[:10]

    return turns

In [10]:
# Load data
train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
dev_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')
test_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)
# print(train_df.columns.tolist())
# print(dev_df.columns.tolist())


train_data = load_convo(train_df)
dev_data = load_convo(dev_df)
test_data = load_convo(test_df)

three_shot = build_prompts(train_data, 3)
five_shot = build_prompts(train_data, 5)

results_3shot = []
results_5shot = []

clean3shot = []
clean5shot = []

dev_preds_3shot = []
dev_preds_5shot = []
dev_refs = []

for convo in dev_data[:50]:
    query = get_turn(convo["turns"][:5], start_num=1)

    # gold turns 6-15 for references later on
    gold_turns = [turn["utterance"] for turn in convo["turns"][5:15]]
    dev_refs.append(" ".join(gold_turns))

    # 3-shot prediction
    prompt3 = build_ICL_prompts(three_shot, query)
    output3 = pipe(
        prompt3,
        **generation_args,
        return_full_text=False, # used for debugging
        max_length=None
    )

    rawoutput3 = output3[0]["generated_text"]
    parsed3 = safe_parse(rawoutput3)

    results_3shot.append(rawoutput3)
    clean3shot.append(parsed3)
    dev_preds_3shot.append(" ".join(parsed3))

    # 5-shot prediction
    prompt5 = build_ICL_prompts(five_shot, query)
    output5 = pipe(
        prompt5,
        **generation_args,
        return_full_text=False, # used for debugging
        max_length=None
    )

    rawoutput5 = output5[0]["generated_text"]
    parsed5 = safe_parse(rawoutput5)

    results_5shot.append(rawoutput5)
    clean5shot.append(parsed5)
    dev_preds_5shot.append(" ".join(parsed5))

print("DONE")
print("3-shot examples:", len(dev_preds_3shot))
print("5-shot examples:", len(dev_preds_5shot))
print("references:", len(dev_refs))

Train: (11090, 12)
Dev: (987, 13)
Test: (2311, 9)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2184 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/ma

DONE
3-shot examples: 33
5-shot examples: 33
references: 33


In [11]:
print(clean3shot[0])
print(len(clean3shot[0])) # should be 10 but better than 5

['I think I would do whatever it took to get out of there.', 'I think I would do whatever it took to get out of there too.', 'Me too.', 'I think I would do whatever it took to get out of there.', 'I think I would do whatever it took to get out of there too.', 'Me too.', 'Me too.', 'Me too.', 'Me too.', 'Me too.']
10


In [12]:
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')

# ROUGE
rouge_3 = rouge.compute(predictions = dev_preds_3shot, references=dev_refs)
rouge_5 = rouge.compute(predictions = dev_preds_5shot, references=dev_refs)
# 3 shot results
print(f"3-shot ROUGE-1: {rouge_3['rouge1']:.4f}")
print(f"3-shot ROUGE-2: {rouge_3['rouge2']:.4f}")
print(f"3-shot ROUGE-L: {rouge_3['rougeL']:.4f}")
print("\n")
# 5 shot results
print(f"5-shot ROUGE-1: {rouge_5['rouge1']:.4f}")
print(f"5-shot ROUGE-2: {rouge_5['rouge2']:.4f}")
print(f"5-shot ROUGE-L: {rouge_5['rougeL']:.4f}")
print("\n")

# BLEU results
bleu_3 = bleu.compute(predictions=dev_preds_3shot, references=[[r] for r in dev_refs])
bleu_5 = bleu.compute(predictions=dev_preds_5shot, references=[[r] for r in dev_refs])

print(f"3-shot BLEU: {bleu_3['bleu']:.4f}")
print(f"5-shot BLEU: {bleu_5['bleu']:.4f}")
print("\n")

#BERT SCORES
bert_3 = bertscore.compute(predictions=dev_preds_3shot, references=dev_refs, lang="en")
bert_5 = bertscore.compute(predictions=dev_preds_5shot, references=dev_refs, lang="en")

# 3-shot bert
print(f"3-Shot Precision {sum(bert_3['precision']) / len(bert_3['precision']):.4f}")
print(f"3-Shot Recall {sum(bert_3['recall']) / len(bert_3['recall']):.4f}")
print(f"3-Shot F1 {sum(bert_3['f1']) / len(bert_3['f1']):.4f}")
print("\n")

# 5-shot bert
print(f"5-Shot Precision {sum(bert_5['precision']) / len(bert_5['precision']):.4f}")
print(f"5-Shot Recall {sum(bert_5['recall']) / len(bert_5['recall']):.4f}")
print(f"5-Shot F1 {sum(bert_5['f1']) / len(bert_5['f1']):.4f}")


3-shot ROUGE-1: 0.2913
3-shot ROUGE-2: 0.0447
3-shot ROUGE-L: 0.1515


5-shot ROUGE-1: 0.2719
5-shot ROUGE-2: 0.0456
5-shot ROUGE-L: 0.1442


3-shot BLEU: 0.0180
5-shot BLEU: 0.0091




Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3-Shot Precision 0.8387
3-Shot Recall 0.8183
3-Shot F1 0.8284


5-Shot Precision 0.8329
5-Shot Recall 0.8118
5-Shot F1 0.8222


In [13]:
rows = []

rows.append({
"Setting": "3-Shot",
"Rouge 1": round(rouge_3["rouge1"], 4),
"Rouge 2": round(rouge_3["rouge2"], 4),
"Rouge L": round(rouge_3["rougeL"],4),
"Bleu": round(bleu_3["bleu"], 4),
"BERT Precision": round(np.mean(bert_3["precision"]), 4),
"BERT Recall": round(np.mean(bert_3["recall"]), 4),
"BERT F1": round(np.mean(bert_3["f1"]), 4)
})

rows.append({
"Setting": "5-Shot",
"Rouge 1": round(rouge_5["rouge1"], 4),
"Rouge 2": round(rouge_5["rouge2"], 4),
"Rouge L": round(rouge_5["rougeL"],4),
"Bleu": round(bleu_5["bleu"], 4),
"BERT Precision": round(np.mean(bert_5["precision"]), 4),
"BERT Recall": round(np.mean(bert_5["recall"]), 4),
"BERT F1": round(np.mean(bert_5["f1"]), 4)
})

analysis_df = pd.DataFrame(rows)
analysis_df.to_csv("q2_dev_metrics.csv", index=False)

In [14]:
submission_rows = []

for convo in test_data:
    query = get_turn(convo["turns"][:5], start_num=1)
    prompt = build_ICL_prompts(three_shot, query)

    output = pipe(
        prompt,
        **generation_args,
        return_full_text=False,
        max_length=None
    )

    pred_turns = safe_parse(output[0]["generated_text"])


    for i in range(10):
      response = pred_turns[i] if i < len(pred_turns) else "I see."
      submission_rows.append({
          "id": convo["id"],
          "turn_number": i + 6,
          "generated_response": response
      })

submission_df = pd.DataFrame(submission_rows)

submission_df.to_csv("q2_test_predictions.csv", index=False)

print(submission_df.head(15))
print(submission_df.shape)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     id  turn_number                                 generated_response
0   393            6  I agree.  I don't think celebrities are that i...
1   393            7  I think most people are the same way.  There i...
2   393            8  Yeah.  I think that the world has its own prob...
3   393            9  I also think that celebrities are often overra...
4   393           10  I think that's true.  People often focus on th...
5   393           11  I think that if people spent more time focusin...
6   393           12  Yeah, I agree.  There are so many problems in ...
7   393           13  I think that people should focus on the things...
8   393           14  I agree.  It's important to stay informed abou...
9   393           15  I think that if people spent more time focusin...
10  394            6  I agree. It is surprising to see the older per...
11  394            7  Perhaps it is because Jolie is so much younger...
12  394            8  That could be a factor. The younger person